In [1]:

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('.env.local')

token = os.getenv('GITHUB_TOKEN')
endpoint = os.getenv('GITHUB_MODELS_ENDPOINT')
deployment = os.getenv('GITHUB_MODELS_DEPLOYMENT')

if not token:
    print("ERROR: GITHUB_TOKEN tidak ditemukan di .env.local")
    exit(1)

print(f"Token (masked): {token[:15]}...{token[-5:]}")
print(f"Endpoint: {endpoint}")
print(f"Default deployment: {deployment}")
print()

client = OpenAI(
    base_url=endpoint,
    api_key=token
)

model_candidates = [
    "openai/gpt-5-mini",
    "gpt-5-mini",
    "openai/gpt-4o-mini",
    "gpt-4o-mini",
]

success_model = None

for model_name in model_candidates:
    print("=" * 60)
    print(f"Testing model: {model_name}")
    print("=" * 60)
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "Kamu asisten AI compliance review Indonesia. Jawab singkat dan presisi."},
                {"role": "user", "content": "Apa itu limbah B3 menurut UU 32/2009? Jawab dalam 1 kalimat."}
            ],
            max_tokens=100,
            temperature=0.3
        )
        print("SUCCESS")
        print(f"Response: {response.choices[0].message.content}")
        print(f"Tokens used: {response.usage.total_tokens}")
        success_model = model_name
        break
    except Exception as e:
        error_msg = str(e)[:300]
        print(f"FAILED: {error_msg}")
        print()

print()
print("=" * 60)
if success_model:
    print(f"SUKSES - Update .env.local jika perlu:")
    print(f"   GITHUB_MODELS_DEPLOYMENT={success_model}")
    print()
    print("LLM layer ComplyScope siap.")
else:
    print("Semua model gagal. Cek:")
    print("1. Token punya permission 'Models: Read-only'")
    print("2. Akun GitHub udah opt-in ke GitHub Models")
print("=" * 60)

Token (masked): github_pat_11BT...NqgP5
Endpoint: https://models.inference.ai.azure.com
Default deployment: openai/gpt-5-mini

Testing model: openai/gpt-5-mini
FAILED: Error code: 400 - {'error': {'code': 'unknown_model', 'message': 'Unknown model: openai/gpt-5-mini', 'details': 'Unknown model: openai/gpt-5-mini'}}

Testing model: gpt-5-mini
FAILED: Error code: 400 - {'error': {'code': 'unavailable_model', 'message': 'Unavailable model: gpt-5-mini', 'details': 'Unavailable model: gpt-5-mini'}}

Testing model: openai/gpt-4o-mini
FAILED: Error code: 400 - {'error': {'code': 'unknown_model', 'message': 'Unknown model: openai/gpt-4o-mini', 'details': 'Unknown model: openai/gpt-4o-mini'}}

Testing model: gpt-4o-mini
SUCCESS
Response: Limbah B3 menurut UU 32/2009 adalah limbah yang mengandung bahan berbahaya dan beracun yang dapat mencemari lingkungan dan membahayakan kesehatan manusia.
Tokens used: 89

SUKSES - Update .env.local jika perlu:
   GITHUB_MODELS_DEPLOYMENT=gpt-4o-mini

LLM layer

In [1]:

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('.env.local')

client = OpenAI(
    base_url=os.getenv('GITHUB_MODELS_ENDPOINT'),
    api_key=os.getenv('GITHUB_TOKEN')
)

MODEL = os.getenv('GITHUB_MODELS_DEPLOYMENT')

# Load corpus
with open('corpus/uu_32_2009.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} entries from corpus")
print()

# Pick 3 pasal yang relevan untuk B3 secara manual (kita validate retrieval nanti)
relevant_pasal_ids = ['uu32_2009_pasal_59_ayat_1', 'uu32_2009_pasal_58_ayat_1', 'uu32_2009_pasal_60_ayat_1']
relevant_pasal = [p for p in corpus if p['id'] in relevant_pasal_ids]

if not relevant_pasal:
    # Fallback: ambil 3 entry yang ada kata "limbah" atau "B3" di teks
    relevant_pasal = [p for p in corpus if 'limbah' in p['teks'].lower() or 'b3' in p['teks'].lower()][:3]

print(f"Using {len(relevant_pasal)} retrieved pasal as context:")
for p in relevant_pasal:
    print(f"  - {p['sitasi']}: {p['teks'][:80]}...")
print()

# SOP dummy text — sengaja gak lengkap untuk test gap detection
sop_text = """
SOP Penanganan Limbah PT Contoh
Bagian 1: Limbah dari proses produksi disimpan di area khusus pabrik.
Bagian 2: Karyawan wajib memakai APD saat menangani limbah.
Bagian 3: Pengangkutan limbah dilakukan setiap minggu oleh pihak ketiga.
"""

# Build prompt
retrieved_context = "\n\n".join([
    f"[{p['sitasi']}]\n{p['teks']}"
    for p in relevant_pasal
])

prompt = f"""Anda adalah asisten review compliance SOP terhadap regulasi Indonesia.

KONTEKS SOP YANG DI-REVIEW:
{sop_text}

PASAL REGULASI YANG DI-RETRIEVE:
{retrieved_context}

TUGAS:
Bandingkan SOP di atas dengan pasal-pasal regulasi.
Identifikasi gap compliance jika ada.

ATURAN KETAT:
1. Hanya gunakan pasal yang SAYA berikan. JANGAN menyebut pasal lain.
2. Jika SOP section tidak terkait dengan topik pasal, abaikan pasal itu.
3. Maksimal 3 gap dalam response ini (untuk test).
4. Output HANYA JSON valid, tanpa markdown.

OUTPUT SCHEMA:
{{
  "status": "COMPLIANT" | "GAP_FOUND" | "NEEDS_REVIEW",
  "summary": "string singkat 1 kalimat",
  "gaps": [
    {{
      "sop_excerpt": "kutipan dari SOP",
      "regulation_ref": "sitasi exact dari pasal",
      "explanation": "penjelasan gap dalam 1-2 kalimat"
    }}
  ]
}}

JSON:"""

print("Sending request to LLM...")
print()

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Kamu asisten compliance review. Output JSON valid saja, tanpa markdown."},
        {"role": "user", "content": prompt}
    ],
    max_tokens=1000,
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
print("=" * 60)
print("RAW LLM OUTPUT:")
print("=" * 60)
print(raw_output)
print()

# Parse JSON
try:
    result = json.loads(raw_output)
    print("=" * 60)
    print("PARSED RESULT:")
    print("=" * 60)
    print(f"Status: {result.get('status')}")
    print(f"Summary: {result.get('summary')}")
    print()
    print(f"Gaps found: {len(result.get('gaps', []))}")
    for i, gap in enumerate(result.get('gaps', []), 1):
        print(f"\n--- Gap #{i} ---")
        print(f"SOP excerpt: {gap.get('sop_excerpt', '')}")
        print(f"Regulation: {gap.get('regulation_ref', '')}")
        print(f"Explanation: {gap.get('explanation', '')}")
    print()
    print("=" * 60)
    print(f"Tokens used: {response.usage.total_tokens}")
    print("=" * 60)
    print()
    print("VALIDATION: Compliance check pipeline working")
except json.JSONDecodeError as e:
    print(f"JSON PARSE ERROR: {e}")
    print("Output bukan JSON valid. Perlu adjust prompt atau temperature.")

Loaded 43 entries from corpus

Using 2 retrieved pasal as context:
  - Pasal 58 ayat (1) UU 32/2009: Setiap orang yang memasukkan ke dalam wilayah Negara Kesatuan Republik Indonesia...
  - Pasal 59 ayat (1) UU 32/2009: Setiap orang yang menghasilkan limbah B3 wajib melakukan pengelolaan limbah B3 y...

Sending request to LLM...

RAW LLM OUTPUT:
{
  "status": "GAP_FOUND",
  "summary": "Terdapat beberapa gap antara SOP dan regulasi yang berlaku.",
  "gaps": [
    {
      "sop_excerpt": "Bagian 1: Limbah dari proses produksi disimpan di area khusus pabrik.",
      "regulation_ref": "Pasal 58 ayat (1) UU 32/2009",
      "explanation": "SOP tidak menjelaskan secara rinci tentang pengelolaan B3 yang harus dilakukan setelah penyimpanan, yang merupakan kewajiban sesuai regulasi."
    },
    {
      "sop_excerpt": "Bagian 2: Karyawan wajib memakai APD saat menangani limbah.",
      "regulation_ref": "Pasal 59 ayat (1) UU 32/2009",
      "explanation": "SOP tidak mencakup prosedur pengelolaan li